# CRML Experiments

Builds the `:experiments` Gradle submodule (shadow JAR), starts a JVM via JPype, and exercises the CRML compiler pipeline directly from Python.

In [1]:
from pathlib import Path

ROOT = Path(".").resolve().parent  # CRML repo root
from experiments.gradle_jvm import GradleJvm

## Build & start JVM

Runs `./gradlew experiments:shadowJar` and starts a JVM with the fat JAR on the classpath.  
The JVM can only be started once per kernel — restart the kernel to rebuild.

In [2]:
jvm = GradleJvm(
    project_path=ROOT,
    subproject="experiments",
    subproject_dir="submodules/experiments",
)
jvm.build()
jvm.start()

import jpype.imports

Running: /home/ubuntu/crml/vol/CRML/gradlew experiments:shadowJar  (cwd=/home/ubuntu/crml/vol/CRML)
Starting a Gradle Daemon, 1 busy and 1 incompatible and 2 stopped Daemons could not be reused, use --status for details
> Task :language:generateGrammarSource

> Task :language:compileJava


3 warnings



> Task :util:generateGrammarSource NO-SOURCE

> Task :util:compileJava


3 warnings



> Task :compiler:compileJava

> Task :compiler:processResources UP-TO-DATE
> Task :compiler:classes


3 warnings


> Task :compiler:jar
> Task :experiments:compileJava NO-SOURCE
> Task :experiments:processResources
> Task :experiments:classes
> Task :language:processResources UP-TO-DATE
> Task :language:classes
> Task :language:jar
> Task :util:processResources NO-SOURCE
> Task :util:classes
> Task :util:jar
> Task :experiments:shadowJar

[Incubating] Problems report is available at: file:///home/ubuntu/crml/vol/CRML/build/reports/problems/problems-report.html

Deprecated Gradle features were used in this build, making it incompatible with Gradle 10.

You can use '--warning-mode all' to show the individual deprecation warnings and determine if they come from your own scripts or plugins.

For more on this, please refer to https://docs.gradle.org/9.1.0/userguide/command_line_interface.html#sec:command_line_warnings in the Gradle documentation.

BUILD SUCCESSFUL in 33s
11 actionable tasks: 9 executed, 2 up-to-date
Consider enabling configuration cache to speed up this build: https://docs.gradle.org/9.

## Seed model validation

Parse the seed models from `experiments.tests.TESTS` and report any syntax errors.  
Seeds are the starting CRML skeletons that each LLM interaction builds upon.

In [3]:
#import jpype
from crml.language.util import Parser
from crml.compiler.omc import OMGenerator
from crml.util import IOUtil

from experiments.tests import TESTS

def parse_seed(name: str, seed: str) -> None:
    result = Parser().parse(seed)
    syntax = result.syntax()
    if syntax.hasErrors():
        errors = [str(e) for e in syntax.errors()]
        print(f"{name}: {len(errors)} error(s)")
        for e in errors:
            print(f"  {e}")
        print(result.toPrettyTree())
    else:
        print(f"{name}: OK")
        code = OMGenerator(result, False);
        print(code.getModelicaCode("fileName"))
    
    

In [ ]:
# SRI domain
parse_seed("SRI", TESTS["SRI"]["seed"])

In [ ]:
# Traffic light domain
parse_seed("traffic", TESTS["traffic"]["seed"])

In [ ]:
# Pumping system domain
parse_seed("pumpsystem", TESTS["pumpsystem"]["seed"])

In [7]:
jvm.shutdown()

JVM shut down.
